# 16S Analyses of the Longitudinal Acne Study
## V1-V3 and V4 primer sets comparison

Date created: 12/28/2024

Notebook authors: Yang Chen

Data analysis by: Yang Chen, Tyler Myers, Britta De Pessemier

This notebook plots the following:

- Plot showing abundance of Cutibacterium in each sample with each primer pair (i.e. the axes are V13 vs V4, each point is the amount of Cutibacterium in one sample by each of the primer pairs)
- Venn diagram illustrating the overlap of genera-level taxa detected by both primer sets, alongside those unique to V1-V3 or V4

In [133]:
# Import Python packages
import pandas as pd
import numpy as np
import biom
import matplotlib.pyplot as plt
import seaborn as sns
import gemelli
from gemelli.preprocessing import rclr_transformation
from matplotlib_venn import venn2, venn2_circles
from Bio import SeqIO
from matplotlib.patches import Circle
from scipy.stats import pearsonr
from upsetplot import UpSet, from_contents
from matplotlib_venn import venn3

In [134]:
# Function to load BIOM table, sort rows by sum
def biom_to_df(biom_path):

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)

    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])

    return df

In [135]:
# Load 16S tables
V1V3_biom = biom_to_df('../Data/16S/Tables/179426_feature-table_16S_V1V3_rare-11054_sampleIDfixed_Genus_SILVA.biom')
V4_biom = biom_to_df('../Data/16S/Tables/174951_feature-table_16S_V4_rare-3769_sampleIDfixed_Genus_SILVA.biom')

In [136]:
# Convert indices to sets for set operations
indices_V1V3 = set(V1V3_biom.index)
indices_V4 = set(V4_biom.index)

# Taxa unique to V1V3
unique_to_V1V3 = indices_V1V3 - indices_V4

# Taxa unique to V4
unique_to_V4 = indices_V4 - indices_V1V3

# Taxa shared by both
shared_taxa = indices_V1V3 & indices_V4

# Step 2: Filter shared and unique taxa based on 10% sample prevalence
def filter_by_prevalence(df, taxa, prevalence_threshold=0.1):
    # Subset the DataFrame to include only the specified taxa
    subset_df = df.loc[list(taxa)]  # Convert set to list for indexing
    # Calculate prevalence: proportion of samples where the taxon is present
    prevalence = (subset_df > 0).sum(axis=1) / subset_df.shape[1]
    # Filter taxa based on the prevalence threshold
    return set(prevalence[prevalence >= prevalence_threshold].index)

# Apply prevalence filtering
filtered_shared = filter_by_prevalence(V1V3_biom, shared_taxa, prevalence_threshold=0.1)
filtered_unique_to_V1V3 = filter_by_prevalence(V1V3_biom, unique_to_V1V3, prevalence_threshold=0.1)
filtered_unique_to_V4 = filter_by_prevalence(V4_biom, unique_to_V4, prevalence_threshold=0.1)

# Step 3: Create the Venn diagram with enhancements
plt.clf()
plt.figure(figsize=(8, 8))  # Increased figure size for better clarity

# Create the Venn diagram with customized circle outlines
venn = venn2(
    [filtered_unique_to_V1V3 | filtered_shared, filtered_unique_to_V4 | filtered_shared],
    ('', ''),
    set_colors=('lightblue', 'lightgreen'),  # Fill colors for the circles
    alpha=0.4  # Transparency for fill colors
)

# Customize the circle outlines with darker colors
for circle, color in zip(['10', '01'], ['blue', 'green']):
    venn.get_patch_by_id(circle).set_edgecolor(color)  # Darker outline color
    venn.get_patch_by_id(circle).set_linewidth(2)  # Thickness of the outline

# Customizing colors for the Venn diagram (switching green and purple)
venn.get_patch_by_id('10').set_color('#87CEEB')  # Light blue for V1-V3
venn.get_patch_by_id('01').set_color('#DDA0DD')  # Light purple for V4
venn.get_patch_by_id('11').set_color('#98FB98')  # Light green for shared

# Customizing text labels
for id in ['10', '01', '11']:
    if venn.get_label_by_id(id):
        venn.get_label_by_id(id).set_fontsize(12)  # Larger font size
        venn.get_label_by_id(id).set_color('black')  # Black text for better readability

# Add a title with larger font size and weight
plt.title('Bacterial Genera Resolved by 16S V1-V3 vs V4',
          fontsize=18)

# Add a legend for the groups
plt.legend(
    handles=[
        plt.Line2D([0], [0], marker='o', color='#87CEEB', lw=0, label='V1-V3 Unique'),
        plt.Line2D([0], [0], marker='o', color='#98FB98', lw=0, label='Shared'),
        plt.Line2D([0], [0], marker='o', color='#DDA0DD', lw=0, label='V4 Unique'),
    ],
    loc='lower center', bbox_to_anchor=(0.5, -0.075), ncol=3, fontsize=12
)

# Save the figure with a higher resolution
plt.savefig('../Figures/Supplementary/Suppl_Figure_7/16S_primer_venn_diagram.png', dpi=600, bbox_inches='tight')

# Show the figure (optional)
plt.show()

# Print the results
print("Filtered Unique to V1V3 (>=10% prevalence):")
print(filtered_unique_to_V1V3)

print("\nFiltered Unique to V4 (>=10% prevalence):")
print(filtered_unique_to_V4)

print("\nFiltered Shared Taxa (>=10% prevalence):")
print(filtered_shared)


Filtered Unique to V1V3 (>=10% prevalence):
{'g__Chloroplast', 'g__Janibacter'}

Filtered Unique to V4 (>=10% prevalence):
{'g__Brachybacterium', 'g__Methylotenera', 'g__Atopobium', 'g__Aggregatibacter', 'g__Phenylobacterium', 'g__Fenollaria', 'g__Thermus', 'g__Vibrio', 'g__Perlucidibaca', 'g__Geobacillus', 'g__Lysobacter', 'g__Peptostreptococcus', 'g__Marinomonas', 'g__Bergeyella', 'g__Empedobacter', 'g__Lactococcus', 'g__Leuconostoc', 'g__Peptoniphilus', 'g__Actinomyces', 'g__Bifidobacterium', 'g__Turicella', 'g__Blautia', 'g__Stenotrophomonas', 'g__Aeromonas', 'g__Paracoccus', 'g__Capnocytophaga', 'g__Acinetobacter', 'g__Chryseobacterium', 'g__Finegoldia', 'g__Moraxella', 'g__Psychrobacter', 'g__Leptotrichia', 'g__Abiotrophia', 'g__Massilia', 'g__Pseudomonas', 'g__Enhydrobacter', 'g__Allorhizobium-Neorhizobium-Pararhizobium-Rhizobium', 'g__Luteimonas', 'g__Bradyrhizobium', 'g__Aerococcus', 'g__Pantoea', 'g__Gardnerella', 'g__Sphingopyxis', 'g__Jeotgalicoccus', 'g__Sediminibacterium'

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_20638/503199925.py:74: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [161]:
# Taxa lists

unique_v1v3 = {
    "Actinobacteriota": ["Janibacter"],
    "Cyanobacteria": ["Chloroplast"]
}

shared_taxa = {
    "Actinobacteriota": [
        "Brevibacterium", "Corynebacterium", "Cutibacterium",
        "Kocuria", "Lawsonella", "Micrococcus", "Nocardioides"
    ],
    "Firmicutes": [
        "Anaerococcus", "Gemella", "Lactobacillus",
        "Rothia", "Staphylococcus", "Streptococcus", "Veillonella"
    ],
    "Bacteroidota": [
        "Alloprevotella", "Porphyromonas", "Prevotella"
    ],
    "Proteobacteria": [
        "Caulobacter", "Haemophilus", "Limnobacter",
        "Neisseria", "Xanthomonas"
    ]
}

unique_v4 = {
    "Actinobacteriota": [
        "Alloiococcus", "Atopobium", "Bifidobacterium",
        "Brachybacterium", "Dolosigranulum", "Gardnerella",
        "Jeotgalicoccus", "Turicella"
    ],
    "Firmicutes": [
        "Abiotrophia", "Aerococcus", "Blautia", "Fenollaria",
        "Finegoldia", "Geobacillus", "Lactococcus",
        "Leuconostoc", "Peptoniphilus", "Peptostreptococcus"
    ],
    "Proteobacteria": [
        "Aeromonas", "Aggregatibacter", "Bradyrhizobium",
        "Enhydrobacter", "Frederiksenia", "Marinomonas",
        "Massilia", "Moraxella", "Pantoea", "Paracoccus",
        "Pseudomonas", "Psychrobacter", "Sphingopyxis",
        "Stenotrophomonas", "Vibrio"
    ],
    "Bacteroidota": [
        "Capnocytophaga", "Chryseobacterium",
        "Empedobacter", "Luteimonas"
    ],
    "Fusobacteriota": [
        "Fusobacterium", "Leptotrichia"
    ]
}

# Helper function to draw a section with tight column alignment

def draw_section(ax, title, taxa_dict, y_start, header_color):
    ax.text(
        0.5, y_start,
        title,
        ha="center",
        va="center",
        fontsize=14,
        fontweight="bold",
        bbox=dict(facecolor=header_color, edgecolor="none", pad=6)
    )

    x_positions = [0.12, 0.42, 0.72]
    y_positions = [y_start - 0.06] * 3
    col = 0

    for phylum, genera in taxa_dict.items():
        x = x_positions[col]
        y = y_positions[col]

        ax.text(
            x, y,
            phylum,
            ha="left",
            va="top",
            fontsize=12,
            fontweight="bold"
        )
        y -= 0.028

        for genus in sorted(genera):
            ax.text(
                x, y,
                genus,
                ha="left",
                va="top",
                fontsize=11,
                style="italic"
            )
            y -= 0.023

        y -= 0.018
        y_positions[col] = y
        col = (col + 1) % 3

    return min(y_positions)

# Create figure

fig = plt.figure(figsize=(8, 13))
ax = fig.add_subplot(111)
ax.axis("off")

y = 0.94
y = draw_section(
    ax,
    "Unique to V1–V3",
    unique_v1v3,
    y,
    header_color="#d9eef7"
)

y -= 0.05
y = draw_section(
    ax,
    "Shared Taxa",
    shared_taxa,
    y,
    header_color="#d9f7d9"
)

y -= 0.05
draw_section(
    ax,
    "Unique to V4",
    unique_v4,
    y,
    header_color="#f2dcf7"
)

# Save figure

out_path = "../Figures/Supplementary/Suppl_Figure_7/Fig_V1V3_V4_taxa_names.png"
plt.savefig(out_path, dpi=600, bbox_inches="tight")
plt.close()

print(f"Saved figure to {out_path}")

Saved figure to ../Figures/Supplementary/Suppl_Figure_7/Fig_V1V3_V4_taxa_names.png


### 3 way venn diagram between 16S V1-V3, 16S V4, and metagenomics from WoLr2

In [137]:
# Load metaG tables
wolr2_biom = biom_to_df('../Data/metaG/Tables/metaG_wolr2_micov-filt_genus.biom')
wolr2_biom.index = wolr2_biom.index.str.replace(r"^ +", "", regex=True)

# rs210_biom = biom_to_df('../Data/metaG/Tables/metaG_rs210_micov-filt_genus.biom')
# rs210_biom.index = rs210_biom.index.str.replace(r"^ +", "", regex=True)

In [158]:
# Plot 3 Venns
set_v1v3  = set(V1V3_biom.index)
set_v4    = set(V4_biom.index)
set_wolr2 = set(wolr2_biom.index)

# Brighter, saturated colors (same hue family as your original)
colors = {
    "v1v3":  "#d0ebf7",  # blue
    "v4":    "#f9cb9c",  # orange
    "wolr2": "#b4a7d6"   # purple
}

plt.figure(figsize=(6, 6))

venn = venn3(
    subsets=(set_v1v3, set_v4, set_wolr2),
    set_labels=("V1–V3", "V4", "metaG WoLr2"),
    set_colors=(colors["v1v3"], colors["v4"], colors["wolr2"]),
    alpha=0.65
)

# Improve label readability
for text in venn.subset_labels:
    if text:
        text.set_fontsize(11)

for text in venn.set_labels:
    text.set_fontsize(11)

plt.title("Shared Genera across 16S and metaG", fontsize=15)
plt.tight_layout()

plt.savefig('../Figures/Main/Figure_4/16S_metaG_venn.png', dpi=600)

### Cutibacterium 16S V1-V3 vs 16S V4 per sample ranked abundance comparison

In [139]:
# Extract the row corresponding to 'g__Cutibacterium' for V1-V3
V1_V3_cutibacterium_df = V1V3_biom.loc[['g__Cutibacterium']]

# Rename the row index
V1_V3_cutibacterium_df.index = ['g__Cutibacterium V1-V3']

# Transpose the DataFrame
V1_V3_cutibacterium_df = V1_V3_cutibacterium_df.T

# Display the transformed DataFrame
V1_V3_cutibacterium_df

,g__Cutibacterium V1-V3
LAMI.RD001.D0.C1,7219.0
LAMI.RD001.D0.C2,5374.0
LAMI.RD001.D14.C1,6592.0
LAMI.RD001.D28.C1,7066.0
LAMI.RD002.D14.C1,10113.0
...,...
LAMI.RD318.D28.C3,9420.0
LAMI.RD319.D14.C1,5440.0
LAMI.RD319.D14.C2,6887.0
LAMI.RD319.D9.C1,3551.0


In [140]:
# Extract the row corresponding to 'g__Cutibacterium' for V4
V4_cutibacterium_df = V4_biom.loc[['g__Cutibacterium']]

# Rename the row index
V4_cutibacterium_df.index = ['g__Cutibacterium V4']

# Transpose the DataFrame
V4_cutibacterium_df = V4_cutibacterium_df.T

# Display the transformed DataFrame
V4_cutibacterium_df

,g__Cutibacterium V4
LAMI.RD001.D0.C1,1.0
173620.14901.BLANK.LOREAL.2.2H,0.0
LAMI.RD001.D14.C1,0.0
LAMI.RD004.D0.C2,1.0
LAMI.RD001.D0.C2,3.0
...,...
LAMI.RD319.D16.C2,3.0
LAMI.RD319.D28.C1,0.0
LAMI.RD318.D9.C2,10.0
LAMI.RD319.D28.C2,0.0


In [141]:
# Get ranks for V1-V3
V1_V3_cutibacterium_df['V1-V3'] = V1_V3_cutibacterium_df['g__Cutibacterium V1-V3'].rank(method='min').astype(int)
V1_V3_cutibacterium_df

,g__Cutibacterium V1-V3,V1-V3
LAMI.RD001.D0.C1,7219.0,31
LAMI.RD001.D0.C2,5374.0,9
LAMI.RD001.D14.C1,6592.0,18
LAMI.RD001.D28.C1,7066.0,26
LAMI.RD002.D14.C1,10113.0,94
...,...,...
LAMI.RD318.D28.C3,9420.0,58
LAMI.RD319.D14.C1,5440.0,10
LAMI.RD319.D14.C2,6887.0,23
LAMI.RD319.D9.C1,3551.0,4


In [142]:
# Get ranks for V4
V4_cutibacterium_df['V4'] = V4_cutibacterium_df['g__Cutibacterium V4'].rank(method='min').astype(int)
V4_cutibacterium_df

,g__Cutibacterium V4,V4
LAMI.RD001.D0.C1,1.0,19
173620.14901.BLANK.LOREAL.2.2H,0.0,1
LAMI.RD001.D14.C1,0.0,1
LAMI.RD004.D0.C2,1.0,19
LAMI.RD001.D0.C2,3.0,50
...,...,...
LAMI.RD319.D16.C2,3.0,50
LAMI.RD319.D28.C1,0.0,1
LAMI.RD318.D9.C2,10.0,87
LAMI.RD319.D28.C2,0.0,1


In [143]:
# Concatenate the two DataFrames along columns, matching on indexes
# Select specific columns
v1_v3_column = V1_V3_cutibacterium_df['V1-V3']  # Adjust column name if needed
v4_column = V4_cutibacterium_df['V4']  # Adjust column name if needed

# Concatenate the selected columns
combined_cutibacterium_df = pd.concat([v1_v3_column, v4_column], axis=1)

# Rename columns for clarity (optional)
combined_cutibacterium_df.columns = ['V1-V3', 'V4']

# Drop rows with NaN values
combined_cutibacterium_df = combined_cutibacterium_df.dropna()

combined_cutibacterium_df

,V1-V3,V4
LAMI.RD001.D0.C1,31.0,19.0
LAMI.RD001.D0.C2,9.0,50.0
LAMI.RD001.D14.C1,18.0,1.0
LAMI.RD001.D28.C1,26.0,19.0
LAMI.RD006.COSM1,2.0,1.0
...,...,...
LAMI.RD318.D28.C3,58.0,59.0
LAMI.RD319.D14.C1,10.0,19.0
LAMI.RD319.D14.C2,23.0,1.0
LAMI.RD319.D9.C1,4.0,1.0


In [144]:
# Scatter plot
plt.figure(figsize=(6, 6))
plt.scatter(
    combined_cutibacterium_df.iloc[:, 0], 
    combined_cutibacterium_df.iloc[:, 1], 
    color='#ffa505', 
    edgecolor='black',
    linewidth=0.5,
    alpha=0.7, 
    label='Samples'
)

# Extract x and y data
x = combined_cutibacterium_df.iloc[:, 0]
y = combined_cutibacterium_df.iloc[:, 1]

# Compute the best-fit line
m, b = np.polyfit(x, y, 1)

# Compute Pearson correlation coefficient and p-value
r, p_value = pearsonr(x, y)

# Plot the best-fit line with the correlation and p-value in the label
plt.plot(x, m * x + b, color='darkorange', label=f'Pearson Correlation: {r:.2f}\np-value: {p_value:.3g}')

# Title and labels
plt.title('Cutibacterium by Primer Set', fontsize=16)
plt.xlabel('Ranked Relative Abundance (V1-V3)', fontsize=14)
plt.ylabel('Ranked Relative Abundance (V4)', fontsize=14)

# Grid, legend, and save
plt.grid(True)
plt.legend()
plt.savefig('../Figures/Supplementary/Suppl_Figure_7/Cutbacterium_V1V3_vs_V4_correlation_ranks.png', dpi=600)
plt.show()


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_20638/1457294117.py:35: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### Comparison of 16S V1-V3, 16S V4, and shotgun of Cutibacterium per-sample ranked relative abundances with intersectional samples

In [145]:
# Load tables
veba_slcs = pd.read_csv('../Analyses/metaG/VEBA_analysis/output_files/X_mags.with_taxonomy.with_slcs_name.csv')
veba_slcs_subset = veba_slcs.iloc[:, 4:]
veba_slcs_subset = veba_slcs_subset.set_index('name')
veba_slcs_subset.index.name = None

veba_slcs_subset_collapsed = veba_slcs_subset.groupby(veba_slcs_subset.index).sum()
relative_abundance_df = veba_slcs_subset_collapsed.div(veba_slcs_subset_collapsed.sum(axis=0), axis=1)
relative_abundance_df = relative_abundance_df.T
cutibacterium_only = relative_abundance_df[['Cutibacterium acnes']]
cutibacterium_only = cutibacterium_only.rename(columns={'Cutibacterium acnes': 'Cutibacterium acnes Shotgun'})

cutibacterium_only.index = cutibacterium_only.index.str.split('_').str[:4].str.join('_')

cutibacterium_only.index = cutibacterium_only.index.str.replace('_', '.', regex=False)


# Get ranks for shotgun
cutibacterium_only['Shotgun'] = cutibacterium_only['Cutibacterium acnes Shotgun'].rank(method='min').astype(int)
cutibacterium_only

,Cutibacterium acnes Shotgun,Shotgun
LAMI.RD308.D9.C3,0.975484,20
LAMI.RD306.D28.C3,0.802056,4
LAMI.RD310.D16.C2,0.925635,10
LAMI.RD304.D14.C3,0.948995,14
LAMI.RD304.D11.C1,0.943899,12
LAMI.RD306.D14.C3,0.790596,3
LAMI.RD308.D0.C3,0.989375,23
LAMI.RD306.D23.C1,0.961753,15
LAMI.RD308.D23.C2,0.874240,7
LAMI.RD310.D7.C3,0.975159,19


In [146]:
# Get ranks for V1-V3 for samples matching with shotgun
V1_V3_cutibacterium_df_shotgun_subset = V1_V3_cutibacterium_df.loc[V1_V3_cutibacterium_df.index.isin(cutibacterium_only.index)]
V1_V3_cutibacterium_df_shotgun_subset = V1_V3_cutibacterium_df_shotgun_subset[['g__Cutibacterium V1-V3']]

# Get ranks for subsetted V1-V3
V1_V3_cutibacterium_df_shotgun_subset['V1-V3'] = V1_V3_cutibacterium_df_shotgun_subset['g__Cutibacterium V1-V3'].rank(method='min').astype(int)
V1_V3_cutibacterium_df_shotgun_subset

,g__Cutibacterium V1-V3,V1-V3
LAMI.RD304.D14.C3,9912.0,4
LAMI.RD304.D7.C1,9997.0,5
LAMI.RD304.D0.C1,10055.0,7
LAMI.RD306.D0.C3,9647.0,2
LAMI.RD308.D0.C2,11005.0,18
LAMI.RD304.D0.C3,10024.0,6
LAMI.RD308.D0.C3,10992.0,17
LAMI.RD306.D23.C1,10463.0,9
LAMI.RD308.D11.C2,10860.0,13
LAMI.RD304.D11.C1,10273.0,8


In [147]:
# Get ranks for V4 for samples matching with shotgun
V4_cutibacterium_df_shotgun_subset = V4_cutibacterium_df.loc[V4_cutibacterium_df.index.isin(cutibacterium_only.index)]
V4_cutibacterium_df_shotgun_subset = V4_cutibacterium_df_shotgun_subset[['g__Cutibacterium V4']]

# Get ranks for subsetted V1-V3
V4_cutibacterium_df_shotgun_subset['V4'] = V4_cutibacterium_df_shotgun_subset['g__Cutibacterium V4'].rank(method='min').astype(int)
V4_cutibacterium_df_shotgun_subset

,g__Cutibacterium V4,V4
LAMI.RD304.D0.C1,16.0,11
LAMI.RD304.D0.C3,12.0,6
LAMI.RD304.D11.C1,15.0,10
LAMI.RD306.D23.C1,20.0,13
LAMI.RD308.D0.C2,93.0,19
LAMI.RD306.D11.C1,2.0,2
LAMI.RD308.D0.C3,92.0,18
LAMI.RD306.D14.C1,7.0,4
LAMI.RD304.D14.C3,21.0,14
LAMI.RD308.D14.C3,134.0,22


In [148]:
# Concatenate the two DataFrames along columns, matching on indexes
# Select specific columns
shotgun_column = cutibacterium_only['Shotgun']
v1_v3_column = V1_V3_cutibacterium_df_shotgun_subset['V1-V3']
v4_column = V4_cutibacterium_df_shotgun_subset['V4']

# Concatenate the selected columns
combined_cutibacterium_df = pd.concat([v1_v3_column, v4_column, shotgun_column], axis=1)

# Rename columns for clarity (optional)
# combined_cutibacterium_df.columns = ['V1-V3', 'V4']

# Drop rows with NaN values
combined_cutibacterium_df = combined_cutibacterium_df.dropna()

combined_cutibacterium_df

,V1-V3,V4,Shotgun
LAMI.RD304.D14.C3,4.0,14.0,14
LAMI.RD304.D7.C1,5.0,5.0,9
LAMI.RD304.D0.C1,7.0,11.0,6
LAMI.RD308.D0.C2,18.0,19.0,22
LAMI.RD304.D0.C3,6.0,6.0,8
LAMI.RD308.D0.C3,17.0,18.0,23
LAMI.RD306.D23.C1,9.0,13.0,15
LAMI.RD304.D11.C1,8.0,10.0,12
LAMI.RD304.D28.C3,10.0,3.0,18
LAMI.RD306.D28.C3,3.0,6.0,4


In [149]:
## uncomment to make 3 data type correlation figure

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.stats import pearsonr, norm
import numpy as np

# Extract columns
x = combined_cutibacterium_df['V1-V3']
y = combined_cutibacterium_df['V4']
z = combined_cutibacterium_df['Shotgun']

# Compute pairwise Pearson correlations with p-values
r1, p1 = pearsonr(x, y)
r2, p2 = pearsonr(x, z)
r3, p3 = pearsonr(y, z)

# Compute average r
r_avg = np.mean([r1, r2, r3])

# --- Estimate combined p-value using Fisher's Z transformation ---
def fisher_z(r):
    return np.arctanh(r)

def inverse_fisher_z(z):
    return np.tanh(z)

# Convert to Fisher Z values
z1 = fisher_z(r1)
z2 = fisher_z(r2)
z3 = fisher_z(r3)

# Mean of z-scores
z_avg = np.mean([z1, z2, z3])

# Convert back to average r (already done)
# r_avg = inverse_fisher_z(z_avg)

# Standard error for Fisher z (n ≈ number of samples)
n = len(x)
se = 1 / np.sqrt(n - 3)

# Compute z-statistic and p-value
z_stat = z_avg / se
p_avg = 2 * (1 - norm.cdf(abs(z_stat)))  # two-tailed

# Create 3D scatter plot
fig = plt.figure(figsize=(10, 6))  # Wider for spacing
ax = fig.add_subplot(111, projection='3d')

# Scatter points
scatter = ax.scatter(x, y, z, c='orange', s=60, edgecolors='k')

# Axes labels and title
ax.set_xlabel('16S V1-V3 Rank')
ax.set_ylabel('16S V4 Rank')
ax.set_zlabel('Shotgun metaG Rank')
ax.set_title('Correlation of Cutibacterium by Sequencing Method', fontsize=16)

# Format correlation results with p-values
textstr = '\n'.join((
    f'V1-V3 vs. V4:',
    f'    Pearson r = {r1:.2f}, p = {p1:.3g}',
    '',
    f'V1-V3 vs. Shotgun:',
    f'    Pearson r = {r2:.2f}, p = {p2:.3g}',
    '',
    f'V4 vs. Shotgun:',
    f'    Pearson r = {r3:.2f}, p = {p3:.3g}',
    '',
    f'All averaged:',
    f'    Pearson r = {r_avg:.2f}, p = {p_avg:.3g}',
))

# Annotate text, positioned further left
plt.gcf().text(0.01, 0.35, textstr, fontsize=10)

# Diagonal 1:1:1 correlation line
min_val = min(x.min(), y.min(), z.min())
max_val = max(x.max(), y.max(), z.max())

ax.plot([min_val, max_val], [min_val, max_val], [min_val, max_val],
        color='orange', linewidth=2, linestyle='-', label='1:1:1 Correlation Line')

# Add n value (# of matched samples between all three data types)
textstr = 'n = 22 matched samples'
plt.gcf().text(0.01, 0.21, textstr, fontsize=10)


# Draw dashed lines from each point to the 1:1:1 diagonal
def project_to_diag(x, y, z):
    direction = np.array([1, 1, 1]) / np.sqrt(3)
    point = np.array([x, y, z])
    proj_length = np.dot(point, direction)
    return proj_length * direction

for xi, yi, zi in zip(x, y, z):
    x_proj, y_proj, z_proj = project_to_diag(xi, yi, zi)
    ax.plot([xi, x_proj], [yi, y_proj], [zi, z_proj],
            color='gray', linestyle='--', linewidth=0.7)

# Add legend to label the orange line
ax.legend(loc='lower left', bbox_to_anchor=(-0.4, 0.25))

plt.tight_layout()
plt.savefig('../Figures/Main/Figure_4/Cutibacterium_3D_correlation.png', dpi=600)


### Phylogenetic tree

In [150]:
# Load the initial taxonomy file
taxonomy_file_path = '../Taxonomy/174116_taxonomy.tsv'
taxonomy_df = pd.read_csv(taxonomy_file_path, sep='\t')

# Extract the Genus from the Taxon column
taxonomy_df['Genus'] = taxonomy_df['Taxon'].str.extract(r'g__(\w+);?')

# Define the groups for each category
v1v3_unique = {'Janibacter', 'Reyranella', 'Mycobacterium'}
v4_unique = {'Empedobacter', 'Stenotrophomonas', 'Capnocytophaga', 'Aerococcus', 
             'Luteimonas', 'Aggregatibacter', 'Peptostreptococcus', 'Enhydrobacter', 
             'Abiotrophia', 'Bifidobacterium', 'Geobacillus', 'Psychrobacter', 'Massilia', 
             'Bradyrhizobium', 'Peptoniphilus', 'Aeromonas', 'Fusobacterium', 'Fenollaria', 
             'Dolosigranulum', 'Atopobium', 'Finegoldia', 'Chryseobacterium', 'Alloiococcus', 
             'Leptotrichia', 'Jeotgalicoccus', 'Gardnerella', 'Brachybacterium', 'Turicella', 
             'Frederiksenia', 'Lactococcus', 'Leuconostoc', 'Marinomonas', 'Pantoea', 
             'Moraxella', 'Vibrio', 'Blautia'
}
shared_taxa = {'Pseudomonas', 'Rothia', 'Gemella', 'Kocuria', 'Porphyromonas', 'Lysobacter', 
               'Nocardioides', 'Lactobacillus', 'Haemophilus', 'Thermus', 'Micrococcus', 
               'Actinomyces', 'Phenylobacterium', 'Neisseria', 'Paracoccus', 'Streptococcus', 
               'Brevibacterium', 'Veillonella', 'Brevundimonas', 'Anaerococcus', 'Caulobacter', 
               'Alloprevotella', 'Sphingopyxis', 'Corynebacterium', 'Prevotella', 'Staphylococcus', 
               'Lawsonella', 'Cutibacterium', 'uncultured', 'Limnobacter'

}

# Filter ASVs for the specified genera
filtered_df = taxonomy_df[taxonomy_df['Genus'].isin(v1v3_unique | v4_unique | shared_taxa)]

# Group by genus and get the most frequent ASV for each genus as the consensus
consensus_asvs = filtered_df.groupby('Genus')['Feature ID'].apply(lambda x: x.mode().iloc[0]).reset_index()
consensus_asvs.columns = ['Genus', 'Consensus ASV']

# Add group column based on the genus
consensus_asvs['group'] = consensus_asvs['Genus'].apply(
    lambda g: 'V1-V3 unique' if g in v1v3_unique else
              'V4 unique' if g in v4_unique else
              'Shared' if g in shared_taxa else
              'Unknown'
)

# Load the updated taxonomy file for resolving placeholders
taxonomy_df_updated = pd.read_csv(taxonomy_file_path, sep='\t')
taxonomy_df_updated['Genus'] = taxonomy_df_updated['Taxon'].str.extract(r'g__(\w+);?')

# Resolve placeholders using the taxonomy data
for index, row in consensus_asvs.iterrows():
    if row['Consensus ASV'].startswith('Placeholder_ASV'):
        genus = row['Genus']
        matching_asvs = taxonomy_df_updated[taxonomy_df_updated['Genus'] == genus]['Feature ID']
        if not matching_asvs.empty:
            consensus_asvs.at[index, 'Consensus ASV'] = matching_asvs.mode().iloc[0]

# Save the final DataFrame to a CSV file for review
consensus_asvs.to_csv('../Analyses/16S/primer_comparison/consensus_asvs.csv', index=False)


In [151]:
# Extract ASVs from the final_consensus_asvs DataFrame into FASTA format
output_fasta_path = "../Analyses/16S/primer_comparison/consensus_asvs.fasta"

with open(output_fasta_path, "w") as fasta_file:
    for index, row in consensus_asvs.iterrows():
        # Only include rows with valid ASVs (no placeholders)
        if not row['Consensus ASV'].startswith('Placeholder_ASV'):
            fasta_file.write(f">{row['Genus']}\n{row['Consensus ASV']}\n")

print(f"FASTA file created at: {output_fasta_path}")


FASTA file created at: ../Analyses/16S/primer_comparison/consensus_asvs.fasta
